# Multi-Table Metrics

This notebook measures Berka relational quality for `account` → `loan` and cross-table feature trends over the paths `account` → `loan` and `district` → `account` → `loan`. It does not rerun single-table column metrics. Transaction-graph fidelity for `account` → `trans` is evaluated separately in `trans_graph_metrics.ipynb`.

In [2]:
import os
from pathlib import Path


def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/home/coder/vector-bootcamp-synthetic-data')

In [3]:
import json
import pickle
from logging import INFO

import numpy as np
import pandas as pd
from midst_toolkit.common.logger import log
from midst_toolkit.evaluation.quality import CorrelationMatrixDifference
from scipy.stats import ks_2samp

from implementations.tabular_data.evaluation.display_utils import log_metrics
from implementations.tabular_data.evaluation.preprocessing import syntheval_preprocess


ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"
# reldiff_data_root = ROOT / "RelDiff" / "data" / "synthetic" / "berka_reldiff" / "RelDiff"
# reldiff_sample_dirs = sorted(reldiff_data_root.glob("reldiff_*/sample*"))
reldiff_data_root = IMPLEMENTATION_ROOT / "multi_table" / "results" / "RelDiff_synth_data"
reldiff_sample_dirs = sorted(reldiff_data_root.glob("sample*"))
if not reldiff_sample_dirs:
    raise FileNotFoundError(f"No RelDiff synthetic sample found under {reldiff_data_root}")
reldiff_sample_dir = reldiff_sample_dirs[-1]

PARENT_TABLE = "account"
CHILD_TABLE = "loan"
PK = "account_id"
FK = "account_id"
TABLES = ("district", "account", "loan")
RELATIONSHIP_KEYS = {
    ("district", "account"): ("district_id", "district_id"),
    ("account", "loan"): ("account_id", "account_id"),
}


def drop_aux_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drop placeholder and cluster columns added during synthesis."""
    drop = [col for col in df.columns if col == "placeholder" or col.endswith("_cluster")]
    return df.drop(columns=drop, errors="ignore")


def load_domain(table_name: str) -> dict:
    """Load a table domain JSON from the Berka multi-table data directory."""
    with open(base_data_dir / f"{table_name}_domain.json") as f:
        return json.load(f)


def feature_columns(domain: dict) -> tuple[list[str], list[str]]:
    """Split a domain file into numerical and categorical column names."""
    numerical = [name for name, spec in domain.items() if spec["type"] == "continuous"]
    categorical = [name for name, spec in domain.items() if spec["type"] == "discrete"]
    return numerical, categorical


real_tables = {name: pd.read_csv(base_data_dir / f"{name}.csv") for name in TABLES}
with open(base_output_dir / "before_matching" / "synthetic_tables.pkl", "rb") as f:
    clava_tables_raw = pickle.load(f)

synthetic_datasets = {
    "ClavaDDPM": {
        "district": drop_aux_columns(clava_tables_raw[(None, "district")]["df"]),
        "account": drop_aux_columns(clava_tables_raw[("district", "account")]["df"]),
        "loan": drop_aux_columns(clava_tables_raw[("account", "loan")]["df"]),
    },
    "RelDiff": {
        name: pd.read_csv(reldiff_sample_dir / f"{name}.csv") for name in TABLES
    },
}

parent_domain = load_domain(PARENT_TABLE)
child_domain = load_domain(CHILD_TABLE)
domains = {name: load_domain(name) for name in TABLES}

real_parent = real_tables[PARENT_TABLE]
real_child = real_tables[CHILD_TABLE]
synthetic_parent = synthetic_datasets["ClavaDDPM"][PARENT_TABLE]
synthetic_child = synthetic_datasets["ClavaDDPM"][CHILD_TABLE]

log(INFO, f"RelDiff sample: {reldiff_sample_dir}")
for model_name, tables in synthetic_datasets.items():
    log(INFO, f"{model_name} rows: " + ", ".join(f"{name}={len(tables[name])}" for name in TABLES))

INFO :      RelDiff sample: /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/results/RelDiff_synth_data/sample1
INFO :      ClavaDDPM rows: district=77, account=4409, loan=639
INFO :      RelDiff rows: district=77, account=4500, loan=682


## 1) Within-table fidelity (per table)

Run the single-table evaluation metrics independently on each table. This is necessary but not sufficient as a multi-table synthesizer can generate high quality individual table while completely breaking the relationships between them.

Those metrics are in `implementations/tabular_data/evaluation/quality/quality_evaluation_pipeline.ipynb`. This notebook only scores **relationships** between `account` and `loan`.

## 2) Structural / referential integrity

- **Referential integrity**: % of foreign keys resolving to a valid parent key, reported for each edge in the evaluated paths (`district` → `account` and `account` → `loan`).

- **Cardinality shape similarity**: for `account` → `loan`, compare the distribution of child rows per parent (including zeros) between real and synthetic data.

### Referential integrity

For each edge used by the evaluated paths, report the fraction of child foreign keys that exist in the parent primary-key column. Closer to 1 is better.

In [6]:
def referential_integrity(
    parent: pd.DataFrame,
    child: pd.DataFrame,
    pk: str,
    fk: str,
) -> dict[str, float]:
    """Score how often child foreign keys resolve to a parent primary key."""
    valid = child[fk].isin(parent[pk])
    n_child = int(len(child))
    n_invalid = int((~valid).sum())
    return {
        "valid_rate": float(valid.mean()) if n_child else float("nan"),
        "n_invalid": n_invalid,
        "n_child": n_child,
        "n_parent": int(len(parent)),
    }


ri_rows = []
for (parent_name, child_name), (pk, fk) in RELATIONSHIP_KEYS.items():
    edge = f"{parent_name} -> {child_name}"
    ri_rows.append(
        {
            "edge": edge,
            "split": "real",
            **referential_integrity(real_tables[parent_name], real_tables[child_name], pk, fk),
        }
    )
    for model_name, tables in synthetic_datasets.items():
        ri_rows.append(
            {
                "edge": edge,
                "split": model_name,
                **referential_integrity(tables[parent_name], tables[child_name], pk, fk),
            }
        )

ri_table = pd.DataFrame(ri_rows)
log_metrics(
    "REFERENTIAL INTEGRITY (evaluated path edges)",
    {
        f"{row['split']}_{row['edge'].replace(' -> ', '_')}_valid_rate": row["valid_rate"]
        for row in ri_rows
    },
)
ri_table  # noqa: B018

INFO :      
REFERENTIAL INTEGRITY (evaluated path edges)
--------------------------------------------------------------------------------

INFO :      Metric: real_district_account_valid_rate	Score: 1.0
INFO :      Metric: ClavaDDPM_district_account_valid_rate	Score: 1.0
INFO :      Metric: RelDiff_district_account_valid_rate	Score: 1.0
INFO :      Metric: real_account_loan_valid_rate	Score: 1.0
INFO :      Metric: ClavaDDPM_account_loan_valid_rate	Score: 1.0
INFO :      Metric: RelDiff_account_loan_valid_rate	Score: 1.0
INFO :      --------------------------------------------------------------------------------



,edge,split,valid_rate,n_invalid,n_child,n_parent
0,district -> account,real,1.0,0,4500,77
1,district -> account,ClavaDDPM,1.0,0,4409,77
2,district -> account,RelDiff,1.0,0,4500,77
3,account -> loan,real,1.0,0,682,4500
4,account -> loan,ClavaDDPM,1.0,0,639,4409
5,account -> loan,RelDiff,1.0,0,682,4500


### Cardinality shape similarity

For every parent row, count how many child rows point at it (including zeros). Compare those two count distributions with a two-sample Kolmogorov–Smirnov statistic.

- **`ks_distance`**: largest gap between the CDFs. Closer to 0 is better.
- **`ks_complement`**: `1 - ks_distance`. Closer to 1 is better (same orientation as SDV cardinality scores).

In [4]:
def children_per_parent(
    parent: pd.DataFrame,
    child: pd.DataFrame,
    pk: str,
    fk: str,
) -> np.ndarray:
    """Count child rows per parent, including parents with zero children."""
    counts = child.groupby(fk).size()
    return parent[pk].map(counts).fillna(0).to_numpy(dtype=float)


real_card = children_per_parent(real_parent, real_child, PK, FK)
cardinality_rows = []
for model_name, tables in synthetic_datasets.items():
    synth_card = children_per_parent(tables[PARENT_TABLE], tables[CHILD_TABLE], PK, FK)
    ks_distance = float(ks_2samp(real_card, synth_card).statistic)
    cardinality_rows.append(
        {
            "model": model_name,
            "ks_distance": ks_distance,
            "ks_complement": 1.0 - ks_distance,
            "real_mean_children": float(real_card.mean()),
            "synthetic_mean_children": float(synth_card.mean()),
            "real_frac_zero": float((real_card == 0).mean()),
            "synthetic_frac_zero": float((synth_card == 0).mean()),
        }
    )

cardinality_table = pd.DataFrame(cardinality_rows)
log_metrics(
    "CARDINALITY SHAPE (account -> loan)",
    {
        f"{row['model']}_ks_complement": row["ks_complement"]
        for row in cardinality_rows
    },
)
cardinality_table  # noqa: B018

INFO :      
CARDINALITY SHAPE (account -> loan)
--------------------------------------------------------------------------------

INFO :      Metric: ClavaDDPM_ks_complement	Score: 0.9933752677603891
INFO :      Metric: RelDiff_ks_complement	Score: 1.0
INFO :      --------------------------------------------------------------------------------



,model,ks_distance,ks_complement,real_mean_children,synthetic_mean_children,real_frac_zero,synthetic_frac_zero
0,ClavaDDPM,0.006625,0.993375,0.151556,0.144931,0.848444,0.855069
1,RelDiff,0.000000,1.000000,0.151556,0.151556,0.848444,0.848444


## 3) Cross-table (inter-table) trends

For each relationship path, join the tables in path order and compare only the feature correlations between the first and last tables. The notebook evaluates `account` → `loan` at `k = 1` and `district` → `account` → `loan` at `k = 2`, for both ClavaDDPM and RelDiff against the real data. Intermediate tables provide the joins but are not part of the endpoint feature block.

The mixed-correlation machinery is the same as `CorrelationMatrixDifference` in the single-table pipeline (Pearson / Cramér’s V / correlation ratio). `corr_mat_diff` is the Frobenius norm of the endpoint-block difference; `mean_abs_corr_diff` normalizes by the number of feature pairs. Both are distances: closer to 0 is better. Compare models at the same `k`, since the number of endpoint feature pairs can differ between paths.

Source: https://docs.sdv.dev/sdv/evaluation/data-quality

In [7]:
def prefix_table_columns(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """Prefix every column so join keys remain unambiguous across a path."""
    return df.rename(columns={col: f"{table_name}__{col}" for col in df.columns})


def join_table_path(tables: dict[str, pd.DataFrame], path: tuple[str, ...]) -> pd.DataFrame:
    """Join tables in path order, retaining the final table's row grain."""
    if len(path) < 2:
        raise ValueError("A table path must contain at least one relationship")

    joined = prefix_table_columns(tables[path[0]], path[0])
    for parent_name, child_name in zip(path, path[1:]):
        parent_pk, child_fk = RELATIONSHIP_KEYS[(parent_name, child_name)]
        child = prefix_table_columns(tables[child_name], child_name)
        joined = joined.merge(
            child,
            left_on=f"{parent_name}__{parent_pk}",
            right_on=f"{child_name}__{child_fk}",
            how="inner",
        )
    return joined


def cross_table_features(
    joined: pd.DataFrame,
    start_table: str,
    end_table: str,
) -> tuple[pd.DataFrame, list[str], list[str], list[str], list[str]]:
    """Select path endpoint features and return their correlation block columns."""
    start_num, start_cat = feature_columns(domains[start_table])
    end_num, end_cat = feature_columns(domains[end_table])
    start_num = [f"{start_table}__{col}" for col in start_num]
    start_cat = [f"{start_table}__{col}" for col in start_cat]
    end_num = [f"{end_table}__{col}" for col in end_num]
    end_cat = [f"{end_table}__{col}" for col in end_cat]
    numeric_cols = start_num + end_num
    categorical_cols = start_cat + end_cat
    feature_cols = numeric_cols + categorical_cols
    return joined[feature_cols].copy(), numeric_cols, categorical_cols, start_num + start_cat, end_num + end_cat


EVALUATION_PATHS = {
    "1-hop account -> loan": ("account", "loan"),
    "2-hop district -> account -> loan": ("district", "account", "loan"),
}
cross_table_rows = []
cross_table_diffs = {}

for path_label, path in EVALUATION_PATHS.items():
    start_table, end_table = path[0], path[-1]
    real_path_join = join_table_path(real_tables, path)
    real_features, numerical_columns, categorical_columns, start_features, end_features = (
        cross_table_features(real_path_join, start_table, end_table)
    )

    for model_name, tables in synthetic_datasets.items():
        synthetic_path_join = join_table_path(tables, path)
        synthetic_features, _, _, _, _ = cross_table_features(
            synthetic_path_join, start_table, end_table
        )
        for col in categorical_columns:
            real_features[col] = real_features[col].astype(str)
            synthetic_features[col] = synthetic_features[col].astype(str)

        real_encoded, synthetic_encoded = syntheval_preprocess(
            numerical_columns,
            categorical_columns,
            real_features,
            synthetic_features,
        )
        metric = CorrelationMatrixDifference(
            categorical_columns=categorical_columns,
            numerical_columns=numerical_columns,
            compute_mixed_correlations=True,
            do_preprocess=False,
        )
        metric.compute(real_encoded, synthetic_encoded)
        matrices = metric.syntheval_metric.evaluate(mixed_corr=True, return_mats=True)
        real_block = matrices["real_cor_mat"].loc[start_features, end_features]
        synthetic_block = matrices["synt_cor_mat"].loc[start_features, end_features]
        block_diff = real_block - synthetic_block
        cross_table_diffs[(model_name, path_label)] = block_diff
        cross_table_rows.append(
            {
                "model": model_name,
                "path": path_label,
                "k": len(path) - 1,
                "corr_mat_diff": float(
                    np.linalg.norm(block_diff.to_numpy(dtype=float), ord="fro")
                ),
                "mean_abs_corr_diff": float(
                    np.abs(block_diff.to_numpy(dtype=float)).mean()
                ),
                "n_column_pairs": int(block_diff.size),
                "real_joined_rows": int(len(real_path_join)),
                "synthetic_joined_rows": int(len(synthetic_path_join)),
            }
        )

cross_table_results = pd.DataFrame(cross_table_rows).sort_values(["k", "model"])
log_metrics(
    "K-HOP ENDPOINT CORRELATION DIFFERENCE",
    {
        f"{row['model']}_{row['k']}_hop_mean_abs_diff": row["mean_abs_corr_diff"]
        for row in cross_table_rows
    },
)
cross_table_results  # noqa: B018

INFO :      
K-HOP ENDPOINT CORRELATION DIFFERENCE
--------------------------------------------------------------------------------

INFO :      Metric: ClavaDDPM_1_hop_mean_abs_diff	Score: 0.12650344172666092
INFO :      Metric: RelDiff_1_hop_mean_abs_diff	Score: 0.017766684302341264
INFO :      Metric: ClavaDDPM_2_hop_mean_abs_diff	Score: 0.04808792312329171
INFO :      Metric: RelDiff_2_hop_mean_abs_diff	Score: 0.02955795705963104
INFO :      --------------------------------------------------------------------------------



,model,path,k,corr_mat_diff,mean_abs_corr_diff,n_column_pairs,real_joined_rows,synthetic_joined_rows
0,ClavaDDPM,1-hop account -> loan,1,0.905393,0.126503,10,682,639
1,RelDiff,1-hop account -> loan,1,0.090623,0.017767,10,682,682
2,ClavaDDPM,2-hop district -> account -> loan,2,0.571683,0.048088,75,682,639
3,RelDiff,2-hop district -> account -> loan,2,0.307903,0.029558,75,682,682
